<a href="https://colab.research.google.com/github/juanpajaro/aprendizaje_profundo_salud_puj_2026/blob/main/LSTM_Apnea_Sueno.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#sistema
import io
import os

#ingenieria de caracteristicas, entrenamiento de modelos
import keras
from keras import layers
import tensorflow as tf

#visualizacion
import matplotlib.pyplot as plt

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# Define the path to your data directory and batch size
data_dir = '/content/drive/MyDrive/datos_apnea'
batch_size = 32

# This code assumes your data is in CSV files named 'train.csv', 'val.csv', and 'test.csv'
# located directly within the specified data_dir.
# It also assumes you have a 'target' column for the labels.
# You MUST replace 'target_column_name_here' with the actual name of your target/label column.

try:
    print(f"Attempting to load datasets from: {data_dir}")

    # Load training dataset
    train_ds = tf.data.experimental.make_csv_dataset(
        file_pattern=os.path.join(data_dir, 'datos_apnea_train.csv'),
        batch_size=batch_size,
        num_epochs=1,  # Iterate once over the dataset
        shuffle=True,  # Shuffle training data
        label_name='Apnea', # IMPORTANT: Replace with your actual target column name
        field_delim=',', # Assuming comma separated values
        ignore_errors=True, # To handle potential malformed rows gracefully
        header=True # Assumes CSVs have headers
    )
    print("Train dataset loaded.")

    # Load validation dataset
    val_ds = tf.data.experimental.make_csv_dataset(
        file_pattern=os.path.join(data_dir, 'datos_apnea_val.csv'),
        batch_size=batch_size,
        num_epochs=1,
        shuffle=False, # No need to shuffle validation data
        label_name='Apnea', # IMPORTANT: Replace with your actual target column name
        field_delim=',',
        ignore_errors=True,
        header=True
    )
    print("Validation dataset loaded.")

    # Load test dataset
    test_ds = tf.data.experimental.make_csv_dataset(
        file_pattern=os.path.join(data_dir, 'datos_apnea_test.csv'),
        batch_size=batch_size,
        num_epochs=1,
        shuffle=False, # No need to shuffle test data
        label_name='Apnea', # IMPORTANT: Replace with your actual target column name
        field_delim=',',
        ignore_errors=True,
        header=True
    )
    print("Test dataset loaded.")

    # Optional: Print a sample batch to verify
    for features, labels in train_ds.take(1):
        print("\n--- Sample Train Batch ---")
        print("Features keys:", list(features.keys()))
        print("Labels shape:", labels.shape)
        print("Labels sample:", labels.numpy())
        break

except Exception as e:
    print(f"\nError loading datasets: {e}")
    print("Please ensure the following:")
    print(f"1. The directory '{data_dir}' exists and contains 'train.csv', 'val.csv', and 'test.csv'.")
    print("2. The CSV files have a header row.")
    print("3. You have replaced 'target_column_name_here' with the correct name of your label column.")
    print("   If your data is structured differently (e.g., images in subdirectories),")
    print("   you might need to use a different Keras utility like 'image_dataset_from_directory'.")


Instructions for updating:
Use `tf.data.Dataset.ignore_errors` instead.


Attempting to load datasets from: /content/drive/MyDrive/datos_apnea
Train dataset loaded.
Validation dataset loaded.
Test dataset loaded.

--- Sample Train Batch ---
Features keys: ['EnfermedadActual']
Labels shape: (32,)
Labels sample: [1 0 1 1 1 0 0 1 1 0 0 0 1 0 0 0 0 1 1 1 0 1 0 1 0 0 0 1 1 1 1 0]


In [4]:
print(type(train_ds))

<class 'tensorflow.python.data.ops.prefetch_op._PrefetchDataset'>


In [6]:
for example, label in train_ds.take(1):
  print('text: ', example["EnfermedadActual"].numpy())
  print('label: ', label.numpy())

text:  [b"Paciente 65 a\xc3\xb1os . refiere asiste solictando formulaicon de metoprolol zok que no estaba incluido en fomrula de cronicos ''cursa con HTA enf coronaria en seguimiento por el programa de RCv''asintoamtico cardiovasucar''dice cumplir indicaiocnes medicas''no ha requerido asistir a urgencias"
 b'Paciente femenina de 83 a\xc3\xb1os de edad, ingresa sola en apoyo de baston, con antecedente de HTA en manejo con ASA 100 mg dia, rosuvastatina 40 mg dia, esomeprazol 20 mg dia, losartan 50 mg cada 12 horas, quien ingresa el dia de hoy con el fin de solicitar formula medica ya que refiere perdio cita de control cardiovascular, ademas de ello refiere caida de su propia altura con trauma a nivel de region costal izquierda con dolor marcada que limita movimientos, niega otra sintomatologia o deficit aparente.'
 b"MASCULINO 61 A\xc3\x91OS INDEPENDIENTE ACUDE SOLO REFIERE EN CITA ANTEIROR SE ORDENARON RADIOGRAFIAS, SOLO APORTA CD NO HAY REPORTES''EN CITA ANTEIROR SE ORDENO GLCIEMIA EN 

In [7]:
#Este linea demora aprox 10 minutos
max_length = 600
max_tokens = 30000
text_vectorization = layers.TextVectorization(
    max_tokens=max_tokens,
    split="whitespace",
    output_mode="int",
    output_sequence_length=max_length,
)
# Adapt the TextVectorization layer to the 'EnfermedadActual' column
train_ds_texts_only = train_ds.map(lambda x, y: x['EnfermedadActual'])
text_vectorization.adapt(train_ds_texts_only)

sequence_train_ds = train_ds.map(
    lambda x, y: (text_vectorization(x['EnfermedadActual']), y), num_parallel_calls=tf.data.AUTOTUNE
)
sequence_val_ds = val_ds.map(
    lambda x, y: (text_vectorization(x['EnfermedadActual']), y), num_parallel_calls=tf.data.AUTOTUNE
)
sequence_test_ds = test_ds.map(
    lambda x, y: (text_vectorization(x['EnfermedadActual']), y), num_parallel_calls=tf.data.AUTOTUNE
)

# Optimize dataset performance
sequence_train_ds = sequence_train_ds.cache().prefetch(buffer_size=tf.data.AUTOTUNE)
sequence_val_ds = sequence_val_ds.cache().prefetch(buffer_size=tf.data.AUTOTUNE)
sequence_test_ds = sequence_test_ds.cache().prefetch(buffer_size=tf.data.AUTOTUNE)

In [8]:
x, y = next(sequence_test_ds.as_numpy_iterator())
print(y.shape)
print(x.shape)

(32,)
(32, 600)


In [9]:
x

array([[8637,   21, 1022, ...,    0,    0,    0],
       [   7, 4910,  864, ...,    0,    0,    0],
       [   7,    4,   39, ...,    0,    0,    0],
       ...,
       [   7,   21,  111, ...,    0,    0,    0],
       [ 163,    2,  397, ...,    0,    0,    0],
       [   7,    2,  457, ...,    0,    0,    0]])

In [10]:
hidden_dim = 64
model = keras.Sequential(
    [
        layers.Embedding(max_tokens, hidden_dim, name="embedding", mask_zero=True),
        layers.Bidirectional(layers.LSTM(64)),
        layers.Dense(64, activation="relu"),
        layers.Dense(1, activation="sigmoid"),
    ]
)

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

In [11]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    restore_best_weights=True,
    patience=2,
)

In [12]:
history = model.fit(
    sequence_train_ds,
    validation_data=sequence_val_ds,
    epochs=10,
    callbacks=[early_stopping],
)
test_loss, test_acc = model.evaluate(sequence_test_ds)
test_acc

Epoch 1/10
   6732/Unknown 4007s 594ms/step - accuracy: 0.5781 - loss: 0.6809

KeyboardInterrupt: 

In [ ]:
accuracy = history.history["accuracy"]
val_accuracy = history.history["val_accuracy"]
epochs = range(1, len(accuracy) + 1)

plt.plot(epochs, accuracy, "r--", label="Training accuracy")
plt.plot(epochs, val_accuracy, "b", label="Validation accuracy")
plt.title("Training and validation accuracy")
plt.legend()
plt.show()